#Transformation query

In [0]:
query = """
    select 
    row_number() over (order by cc.customer_id) as customer_key
    , cc.customer_id
    , cc.customer_number
    , cc.first_name
    , cc.last_name
    , el.country
    , cc.marital_status
    , case 
        when cc.gender <> 'n/a' then cc.gender
        else coalesce(ec.gender,'n/a')
    end as gender
    , ec.Birth_date as birth_date
    , cc.created_date as created_date
    from workspace.silver.crm_customers cc
    left join workspace.silver.erp_customers ec on cc.customer_number = ec.customer_number
    left join workspace.silver.erp_customer_location el on cc.customer_number = el.customer_number
"""

df = spark.sql(query)

In [0]:
df.limit(10).display()

#writing dataframe to gold table

In [0]:
%sql
drop table if exists workspace.gold.dim_customers

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.gold.dim_customers")

#Sanity Check of gold table

In [0]:
%sql
select * from workspace.gold.dim_customers limit 10